# Diabetes Prediction — End-to-End Pipeline (Part 1)

**Dataset:** BRFSS 2015 Diabetes Health Indicators (`diabetes_binary_health_indicators_BRFSS2015.csv`)
253,680 survey respondents, 21 health/lifestyle features, target = `Diabetes_binary` (1 = diabetic/prediabetic, 0 = not).

This notebook covers the first half of the pipeline:
1. Data Collection
2. Data Understanding (EDA)
3. Data Cleaning
4. Correlation Heatmap

(Feature selection, preprocessing, train/test split, modeling, and evaluation come after this and are not in scope here.)


## Step 0 — Imports
Standard stack: pandas/numpy for data handling, matplotlib/seaborn for visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)


## Step 1 — Data Collection

Load the raw CSV. This is the "collection" step — in a real project this might instead be a SQL query, an API pull, or a hospital records export. Here it's already a flat file, so collection = reading it in.

In [ ]:
df = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")

print("Shape (rows, columns):", df.shape)
df.head()


## Step 2 — Data Understanding (EDA)

Before touching anything, understand what you have:
- What are the columns and their types?
- Any obviously wrong ranges?
- Is the target balanced or imbalanced?
- What do the summary statistics look like?


In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
# Target balance — important because BRFSS is a real-world survey, so diabetics are the minority
target_counts = df["Diabetes_binary"].value_counts()
print(target_counts)
print((target_counts / len(df) * 100).round(2).astype(str) + " %")

target_counts.plot(kind="bar", title="Diabetes_binary class distribution")
plt.xlabel("0 = No diabetes, 1 = Diabetes/prediabetes")
plt.ylabel("Count")
plt.show()


**Observation to note down:** the classes are imbalanced (far more 0s than 1s). That's a real-world detail worth mentioning — it will matter later for modeling (may need class weighting or the pre-balanced 50/50 version of this dataset), but it doesn't change the cleaning or correlation steps below.

## Step 3 — Data Cleaning

Cleaning checklist for any tabular dataset:
1. Missing values
2. Duplicate rows
3. Invalid / impossible values (e.g. a BMI of 0)
4. Correct data types

This dataset is a pre-processed CDC survey export, so it's cleaner than most raw hospital data — but you should still **check**, not assume.

In [ ]:
# 1. Missing values
missing = df.isnull().sum()
print("Columns with missing values:")
print(missing[missing > 0] if missing.sum() > 0 else "None found.")


In [ ]:
# 2. Duplicate rows
dupe_count = df.duplicated().sum()
print(f"Duplicate rows found: {dupe_count}")

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after dropping duplicates:", df.shape)


In [ ]:
# 3. Invalid / impossible values — sanity-check ranges per column
# e.g. BMI should never realistically be 0
print("Rows with BMI == 0:", (df["BMI"] == 0).sum())
print("BMI range:", df["BMI"].min(), "-", df["BMI"].max())

# GenHlth, Age, Education, Income are coded categories (1-5, 1-13, 1-6, 1-8) — confirm no out-of-range codes
for col, valid_range in [("GenHlth", (1, 5)), ("Age", (1, 13)), ("Education", (1, 6)), ("Income", (1, 8))]:
    bad = df[~df[col].between(*valid_range)]
    print(f"{col}: {len(bad)} rows outside expected range {valid_range}")


In [ ]:
# 4. Data types — everything loaded as float64 even though most columns are really binary (0/1) flags or int codes.
# Casting them properly makes downstream analysis and memory usage cleaner.
binary_cols = ["Diabetes_binary", "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
               "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies", "HvyAlcoholConsump",
               "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"]
int_coded_cols = ["GenHlth", "Age", "Education", "Income"]

df[binary_cols] = df[binary_cols].astype(int)
df[int_coded_cols] = df[int_coded_cols].astype(int)

df.dtypes


**Cleaning summary to say out loud:** no missing values, some duplicate rows removed, no impossible values in BMI or the coded categorical columns, and dtypes tightened from float64 to int where the data is really binary/categorical. Data is now ready for exploration.

## Step 4 — Correlation Heatmap

Now that the data is clean, compute the pairwise correlation matrix (Pearson correlation, range -1 to +1) across all features and the target, and visualize it as a heatmap.

**Why this step matters (say this in your explanation):**
- **Feature relevance:** shows which features correlate most strongly (positive or negative) with `Diabetes_binary` — these are your strongest candidate predictors.
- **Multicollinearity check:** shows which *independent* features are highly correlated with *each other* (e.g. `GenHlth` and `PhysHlth`). Redundant features can be dropped or handled carefully before modeling.
- **Sanity check:** correlations should make medical sense (e.g. `HighBP`, `HighChol`, `BMI`, `Age`, `GenHlth` should show positive correlation with diabetes) — confirms the data isn't corrupted.
- **Guides the next step (feature selection)**, which is where this notebook's scope ends.

In [ ]:
corr = df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Correlation Heatmap — All Features + Diabetes_binary")
plt.tight_layout()
plt.show()


In [ ]:
# The full matrix is dense with 22 columns — pull out just the correlations with the target, sorted
target_corr = corr["Diabetes_binary"].drop("Diabetes_binary").sort_values(key=abs, ascending=False)
print("Feature correlation with Diabetes_binary (strongest first):")
print(target_corr)

plt.figure(figsize=(8, 8))
sns.heatmap(target_corr.to_frame(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation of each feature with Diabetes_binary")
plt.tight_layout()
plt.show()


## Recap

- **Collected** the BRFSS2015 diabetes health indicators dataset.
- **Understood** it: 253k rows, 21 features + target, imbalanced classes, mostly binary/categorical coded features.
- **Cleaned** it: checked and confirmed no missing values, removed duplicate rows, validated value ranges, fixed dtypes.
- **Correlation heatmap**: identified `GenHlth`, `HighBP`, `BMI`, `HighChol`, `Age`, `DiffWalk` as the features most correlated with diabetes — these are the strongest candidates to carry into feature selection and modeling.

**Next steps (not covered in this notebook):** feature selection/scaling, train/test split, model training (Logistic Regression, Random Forest, etc.), and evaluation (precision/recall/F1/ROC-AUC, given the class imbalance).